Collect one real source → adapt it → run CRIMEX pipeline → save first enriched CRIMEX dataset

# CRIMEX v1 — First Real Dataset Build

Goal:
Download one real crime source, standardize it, run the CRIMEX enrichment pipeline, and save the first enriched dataset.

## 1. Setup

Prepare imports, folders, and project paths.

In [1]:
# [1.1] Setup

import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

RAW_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Allow importing local src modules
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Final folder:", FINAL_DIR)

Project root: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\notebooks
Raw folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\notebooks\data\raw
Final folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\notebooks\data\final


## 2. Download LA Crime Open Data

Download a real incident-level sample from LA Open Data.

In [9]:
# [2.1] Download full LA crime dataset with pagination

import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

base_url = "https://data.lacity.org/resource/2nrs-mtv8.csv"

limit = 50000
offset = 0
chunks = []

while True:
    url = f"{base_url}?$limit={limit}&$offset={offset}"
    chunk = pd.read_csv(url)

    print(f"Downloaded offset {offset}: {len(chunk)} rows")

    if chunk.empty:
        break

    chunks.append(chunk)

    if len(chunk) < limit:
        break

    offset += limit

df_raw = pd.concat(chunks, ignore_index=True)

raw_path = RAW_DIR / "la_crime_full_raw.csv"
df_raw.to_csv(raw_path, index=False)

print("Final shape:", df_raw.shape)
print("Saved:", raw_path)

df_raw.head()

Downloaded offset 0: 50000 rows
Downloaded offset 50000: 50000 rows
Downloaded offset 100000: 50000 rows
Downloaded offset 150000: 50000 rows
Downloaded offset 200000: 50000 rows
Downloaded offset 250000: 50000 rows
Downloaded offset 300000: 50000 rows
Downloaded offset 350000: 50000 rows
Downloaded offset 400000: 50000 rows
Downloaded offset 450000: 50000 rows
Downloaded offset 500000: 50000 rows
Downloaded offset 550000: 50000 rows
Downloaded offset 600000: 50000 rows
Downloaded offset 650000: 50000 rows
Downloaded offset 700000: 50000 rows
Downloaded offset 750000: 50000 rows
Downloaded offset 800000: 50000 rows
Downloaded offset 850000: 50000 rows
Downloaded offset 900000: 50000 rows
Downloaded offset 950000: 50000 rows
Downloaded offset 1000000: 4894 rows
Final shape: (1004894, 28)
Saved: data\raw\la_crime_full_raw.csv


,dr_no,date_rptd,date_occ,time_occ,area,area_name,rpt_dist_no,part_1_2,crm_cd,crm_cd_desc,...,status,status_desc,crm_cd_1,crm_cd_2,crm_cd_3,crm_cd_4,location,cross_street,lat,lon
0,211507896,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


## 3. Basic integrity checks

Check whether case IDs are unique and whether obvious duplicates exist.

In [10]:
# [3.1] Basic integrity checks

print("Rows:", len(df_raw))

print(
    "Unique case_ids:",
    df_raw["dr_no"].nunique()
)

print(
    "Duplicate case_ids:",
    len(df_raw) - df_raw["dr_no"].nunique()
)

Rows: 1004894
Unique case_ids: 1004894
Duplicate case_ids: 0


## 3.2 Check missingness in core CRIMEX input fields

In [11]:
# [3.2] Missingness for key source fields

core_fields = [
    "dr_no",
    "crm_cd_desc",
    "date_occ",
    "time_occ",
    "lat",
    "lon",
    "mocodes",
    "weapon_desc",
    "premis_desc",
    "vict_age",
    "vict_sex",
    "vict_descent"
]

(
    df_raw[core_fields]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

weapon_desc     677678
mocodes         151598
vict_descent    144643
vict_sex        144631
premis_desc        588
dr_no                0
crm_cd_desc          0
date_occ             0
time_occ             0
lat                  0
lon                  0
vict_age             0
dtype: int64

## 4. Standardize source fields

Map raw source columns into CRIMEX input schema.

In [12]:
# [4.1] Map source into CRIMEX input schema

LA_MAPPING = {
    "dr_no":"case_id",
    "crm_cd_desc":"crime_description",
    "date_rptd":"date_reported",
    "date_occ":"date_occured",
    "time_occ":"time_occured",
    "lat":"latitude",
    "lon":"longitude",
    "mocodes":"modus_operandi",
    "weapon_desc":"weapon_description",
    "premis_desc":"premise_description",
    "vict_age":"victim_age",
    "vict_sex":"victim_sex",
    "vict_descent":"victim_ethnicity"
}

df_input = df_raw.rename(
    columns=LA_MAPPING
).copy()

# minimal source provenance
df_input["source_system_code"] = "la_open_data"
df_input["source_country_code"] = "US"

print(
    "Input shape:",
    df_input.shape
)

# verify core mapped columns
mapped_cols = list(LA_MAPPING.values())
print(mapped_cols)

df_input[
    mapped_cols[:8]
].head()

Input shape: (1004894, 30)
['case_id', 'crime_description', 'date_reported', 'date_occured', 'time_occured', 'latitude', 'longitude', 'modus_operandi', 'weapon_description', 'premise_description', 'victim_age', 'victim_sex', 'victim_ethnicity']


,case_id,crime_description,date_reported,date_occured,time_occured,latitude,longitude,modus_operandi
0,211507896,THEFT OF IDENTITY,2021-04-11T00:00:00.000,2020-11-07T00:00:00.000,845,34.2124,-118.4092,0377
1,201516622,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",2020-10-21T00:00:00.000,2020-10-18T00:00:00.000,1845,34.1993,-118.4203,0416 0334 2004 1822 1414 0305 0319 0400
2,240913563,THEFT OF IDENTITY,2024-12-10T00:00:00.000,2020-10-30T00:00:00.000,1240,34.1847,-118.4509,0377
3,210704711,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,2020-12-24T00:00:00.000,2020-12-24T00:00:00.000,1310,34.0339,-118.3747,0344
4,201418201,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),2020-10-03T00:00:00.000,2020-09-29T00:00:00.000,1830,33.9813,-118.4350,1300 0344 1606 2032


## 4.2 Basic input cleaning

Clean dates, time field, text normalization, and missing categorical values before enrichment.

In [13]:
# [4.2] Basic cleaning

import pandas as pd

# Dates
df_input["date_occured"] = pd.to_datetime(
    df_input["date_occured"],
    errors="coerce"
)

df_input["date_reported"] = pd.to_datetime(
    df_input["date_reported"],
    errors="coerce"
)

# Standardize crime text
df_input["crime_description"] = (
    df_input["crime_description"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# preserve unknown explicitly
for col in [
    "weapon_description",
    "premise_description",
    "victim_sex",
    "victim_ethnicity"
]:
    df_input[col] = (
        df_input[col]
        .fillna("unknown")
    )

# MO codes as string
df_input["modus_operandi"] = (
    df_input["modus_operandi"]
    .fillna("")
    .astype(str)
)

# time field padding (845 -> 0845)
df_input["time_occured"] = (
    df_input["time_occured"]
    .astype(str)
    .str.zfill(4)
)

print("Cleaning complete")

df_input[
[
"crime_description",
"time_occured",
"weapon_description",
"victim_sex"
]
].head()

Cleaning complete


,crime_description,time_occured,weapon_description,victim_sex
0,theft of identity,0845,unknown,M
1,"assault with deadly weapon, aggravated assault",1845,KNIFE WITH BLADE 6INCHES OR LESS,M
2,theft of identity,1240,unknown,M
3,theft from motor vehicle - grand ($950.01 and ...,1310,unknown,F
4,theft from motor vehicle - petty ($950 & under),1830,unknown,M


## 5.1 Run full CRIMEX enrichment pipeline

Run the full CRIMEX enrichment pipeline with all agreed feature pillars enabled.

In [ ]:
# [5.0] Expected CRIMEX schema registry (first pass)

EXPECTED_CRIMEX_PILLARS = [
"source_provenance",
"incident_core",
"temporal",
"geo",
"behavior",
"context",
"ontology",
"quality_validation",
"graph_intelligence",
"explainability",
"pipeline_metadata"
]

EXPECTED_CRIMEX_PILLARS

In [16]:
import importlib
import src.behavior
importlib.reload(src.behavior)

print("behavior.py reloaded")

behavior.py reloaded


In [17]:
import importlib
import src.context
importlib.reload(src.context)

print("context.py reloaded")

context.py reloaded


In [18]:
import importlib
import src.graph
importlib.reload(src.graph)

print("graph.py reloaded")

graph.py reloaded


In [19]:
import importlib
import src.pipeline
importlib.reload(src.pipeline)

print("pipeline.py reloaded")

pipeline.py reloaded


In [22]:
import importlib
import src.quality
importlib.reload(src.quality)

print("pipeline.py reloaded")

pipeline.py reloaded


In [24]:
def run_crimex_pipeline(df, pipeline_config):

    df = df.copy()

    print("Step 1/9 → Schema standardization")
    df = standardize_schema(df)

    print("Step 2/9 → Cleaning")
    df = basic_cleaning_pipeline(df)

    print("Step 3/9 → Multilingual / ontology")
    if pipeline_config.get("enable_translation_module", True):
        df = multilingual_pipeline(df)

    print("Step 4/9 → Geo validation")
    df = validate_coordinates(df)

    print("Step 5/9 → Temporal features")
    df = add_temporal_features(df)

    print("Step 6/9 → Behavioral features (heavy)")
    df = add_behavior_features(df)

    print("Step 7/9 → Context features (heavy)")
    df = add_context_features(df)

    print("Step 8/9 → Quality features")
    if pipeline_config.get("enable_quality_validation", True):
        df = add_quality_features(df)

    print("Step 9/9 → Graph + Explainability")
    if pipeline_config.get("enable_graph_features", True):
        df = add_graph_intelligence_features(df)

    df = add_explainability_features(df)

    print("Pipeline completed")

    return df

In [26]:
# [5.0] Install tqdm safely if missing

import sys
import subprocess

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "tqdm"
    ])
    from tqdm.auto import tqdm

print("tqdm is ready")

tqdm ready


In [30]:
# [5.1] Reload pipeline and run

import importlib
import src.pipeline
importlib.reload(src.pipeline)

from src.config import PIPELINE_CONFIG
from src.pipeline import run_crimex_pipeline

PIPELINE_CONFIG.update({
    "enable_geo_enrichment": False,
    "enable_poi_enrichment": False,
    "enable_translation_module": True,
    "enable_weather_enrichment": True,
    "enable_graph_features": True,
    "enable_actor_enrichment": True,
    "enable_quality_validation": True
})

df_crimex = run_crimex_pipeline(
    df_input,
    PIPELINE_CONFIG
)

df_crimex.head()


Explainability: 100%|█████████████████████████████████████████████████████████████| 10/10 [2:11:41<00:00, 790.12s/step]

CRIMEX pipeline completed.
Final shape: (1004894, 156)


,case_id,date_reported,date_occured,time_occured,area,area_name,rpt_dist_no,part_1_2,crm_cd,crime_description,...,behavior_centrality_risk_score,behavior_centrality_risk_code,location_connectivity_degree,location_connectivity_code,cooffending_risk_score,cooffending_risk_code,network_exposure_risk_score,network_exposure_risk_code,risk_explanation_text,risk_explanation_signature_code
0,211507896,2021-04-11,2020-11-07,08:45:00,15,N Hollywood,1502,2,354,theft of identity,...,3,high,6,high,1,low,5,high,elevated network exposure risk,NETWORK
1,201516622,2020-10-21,2020-10-18,18:45:00,15,N Hollywood,1521,1,230,"assault with deadly weapon, aggravated assault",...,3,high,15,high,5,high,9,high,high behavioral threat | high routine activity...,THREAT|ROUTINE|NETWORK
2,240913563,2024-12-10,2020-10-30,12:40:00,9,Van Nuys,933,2,354,theft of identity,...,3,high,44,high,1,low,5,high,elevated network exposure risk,NETWORK
3,210704711,2020-12-24,2020-12-24,13:10:00,7,Wilshire,782,1,331,theft from motor vehicle - grand ($950.01 and ...,...,3,high,27,high,1,low,5,high,elevated network exposure risk,NETWORK
4,201418201,2020-10-03,2020-09-29,18:30:00,14,Pacific,1454,1,420,theft from motor vehicle - petty ($950 & under),...,3,high,25,high,3,medium,7,high,elevated network exposure risk,NETWORK


In [31]:
# [6.1] Schema validation

expected_columns = 156  # your expected

print("Actual columns:", len(df_crimex.columns))

missing = []
# add manually any must-have columns you expect

print("Done")

Actual columns: 156
Done


In [33]:
# [6.2] Save CRIMEX dataset

df_crimex.to_parquet(
    "crimex_la_full.parquet",
    index=False
)

print("Saved successfully")

Saved successfully


df_crimex
→ extract unique coordinates
→ enrich coordinates in batches
→ save checkpoint after each batch
→ merge results back to df_crimex

In [34]:
# [6.1] Extract unique valid coordinates

geo_keys = [
    "latitude_final",
    "longitude_final"
]

unique_coords = (
    df_crimex[
        df_crimex["is_valid_coordinate"] == 1
    ][geo_keys]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Total rows:", len(df_crimex))
print("Unique valid coordinates:", len(unique_coords))

unique_coords.head()

Total rows: 1004894
Unique valid coordinates: 77868


,latitude_final,longitude_final
0,34.2124,-118.4092
1,34.1993,-118.4203
2,34.1847,-118.4509
3,34.0339,-118.3747
4,33.9813,-118.4350


## 6.2 Identify rows missing coordinates

We separate rows that need fallback geocoding.

In [36]:
# [6.2] Missing coordinates - safe version

missing_geo = df_crimex[
    df_crimex["is_valid_coordinate"] == 0
]

print("Missing geo rows:", len(missing_geo))

candidate_cols = [
    "area_name",
    "location",
    "street_address",
    "cross_street",
    "source_country_code"
]

existing_cols = [
    col for col in candidate_cols
    if col in df_crimex.columns
]

print("Available columns:", existing_cols)

missing_geo[
    existing_cols
].head()

Missing geo rows: 2240
Available columns: ['area_name', 'location', 'cross_street', 'source_country_code']


,area_name,location,cross_street,source_country_code
346,N Hollywood,LANKERSHIM,VINELAND,US
533,Pacific,300 WORLD WY,NaN,US
560,Wilshire,1000 QUEEN ANNE PL,NaN,US
591,Devonshire,9400 RESEDA BL,NaN,US
1085,Southeast,10500 PARMELEE AV,NaN,US


## 6.3 Prepare fallback geocoding input

We build address strings for missing coordinates using available fields.

In [37]:
# [6.3] Build address for geocoding

missing_geo = missing_geo.copy()

def build_address(row):
    parts = []

    if pd.notna(row.get("location")):
        parts.append(str(row["location"]))

    if pd.notna(row.get("cross_street")):
        parts.append(str(row["cross_street"]))

    if pd.notna(row.get("area_name")):
        parts.append(str(row["area_name"]))

    parts.append("Los Angeles USA")

    return ", ".join(parts)

missing_geo["geo_query"] = missing_geo.apply(
    build_address,
    axis=1
)

missing_geo[
    ["geo_query"]
].head()

,geo_query
346,"LANKERSHIM, VINELAND, N Hollywood, Los Angeles..."
533,"300 WORLD WY, Pacifi..."
560,"1000 QUEEN ANNE PL, Wilsh..."
591,"9400 RESEDA BL, Devon..."
1085,"10500 PARMELEE AV, Sout..."


In [ ]:
## 6.4 Batch geocoding for missing coordinates (with cache)

In [38]:
# [6.4] Batch geocoding with caching

import time
import requests
import pandas as pd
from pathlib import Path

CACHE_PATH = Path("data/cache/geocoding_cache.csv")
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

# Load cache if exists
if CACHE_PATH.exists():
    geo_cache = pd.read_csv(CACHE_PATH)
else:
    geo_cache = pd.DataFrame(columns=["geo_query", "lat", "lon"])

cache_dict = dict(zip(geo_cache["geo_query"], zip(geo_cache["lat"], geo_cache["lon"])))

results = []

batch_size = 50

for i, row in missing_geo.iterrows():

    query = row["geo_query"]

    # Use cache
    if query in cache_dict:
        lat, lon = cache_dict[query]
    else:
        try:
            url = "https://nominatim.openstreetmap.org/search"

            params = {
                "q": query,
                "format": "json",
                "limit": 1
            }

            headers = {
                "User-Agent": "CRIMEX-Geocoder"
            }

            response = requests.get(url, params=params, headers=headers)

            data = response.json()

            if len(data) > 0:
                lat = float(data[0]["lat"])
                lon = float(data[0]["lon"])
            else:
                lat, lon = None, None

            # Respect rate limit
            time.sleep(1)

        except Exception:
            lat, lon = None, None

        # update cache
        cache_dict[query] = (lat, lon)

    results.append((query, lat, lon))

    # Save checkpoint every batch
    if len(results) % batch_size == 0:
        print(f"Processed {len(results)} / {len(missing_geo)}")

        temp_df = pd.DataFrame(results, columns=["geo_query", "lat", "lon"])
        temp_df.to_csv(CACHE_PATH, index=False)

# Final save
geo_results = pd.DataFrame(results, columns=["geo_query", "lat", "lon"])
geo_results.to_csv(CACHE_PATH, index=False)

print("Geocoding completed")
geo_results.head()

Processed 50 / 2240
Processed 100 / 2240
Processed 150 / 2240
Processed 200 / 2240
Processed 250 / 2240
Processed 300 / 2240
Processed 350 / 2240
Processed 400 / 2240
Processed 450 / 2240
Processed 500 / 2240
Processed 550 / 2240
Processed 600 / 2240
Processed 650 / 2240
Processed 700 / 2240
Processed 750 / 2240
Processed 800 / 2240
Processed 850 / 2240
Processed 900 / 2240
Processed 950 / 2240
Processed 1000 / 2240
Processed 1050 / 2240
Processed 1100 / 2240
Processed 1150 / 2240
Processed 1200 / 2240
Processed 1250 / 2240
Processed 1300 / 2240
Processed 1350 / 2240
Processed 1400 / 2240
Processed 1450 / 2240
Processed 1500 / 2240
Processed 1550 / 2240
Processed 1600 / 2240
Processed 1650 / 2240
Processed 1700 / 2240
Processed 1750 / 2240
Processed 1800 / 2240
Processed 1850 / 2240
Processed 1900 / 2240
Processed 1950 / 2240
Processed 2000 / 2240
Processed 2050 / 2240
Processed 2100 / 2240
Processed 2150 / 2240
Processed 2200 / 2240
Geocoding completed


,geo_query,lat,lon
0,"LANKERSHIM, VINELAND, N Hollywood, Los Angeles...",34.158448,-118.370616
1,"300 WORLD WY, Pacifi...",NaN,NaN
2,"1000 QUEEN ANNE PL, Wilsh...",34.055654,-118.329759
3,"9400 RESEDA BL, Devon...",NaN,NaN
4,"10500 PARMELEE AV, Sout...",NaN,NaN


In [39]:
# [6.5] Check geocoding success rate

geocoded_count = geo_results[["lat", "lon"]].notna().all(axis=1).sum()
failed_count = len(geo_results) - geocoded_count

print("Total attempted:", len(geo_results))
print("Geocoded:", geocoded_count)
print("Failed:", failed_count)
print("Success rate:", round(geocoded_count / len(geo_results) * 100, 2), "%")

geo_results.head(10)

Total attempted: 2240
Geocoded: 452
Failed: 1788
Success rate: 20.18 %


,geo_query,lat,lon
0,"LANKERSHIM, VINELAND, N Hollywood, Los Angeles...",34.158448,-118.370616
1,"300 WORLD WY, Pacifi...",NaN,NaN
2,"1000 QUEEN ANNE PL, Wilsh...",34.055654,-118.329759
3,"9400 RESEDA BL, Devon...",NaN,NaN
4,"10500 PARMELEE AV, Sout...",NaN,NaN
5,"ORO VISTA, GROVE, Foothill, Los Angeles USA",NaN,NaN
6,"300 WORLD WY, Pacifi...",NaN,NaN
7,"700 W IMPERIAL HY, Southe...",NaN,NaN
8,"OLYMPIC BL, MANHATTAN ...",NaN,NaN
9,"300 E 99TH ST, Southe...",NaN,NaN


In [42]:
# [6.6] Merge successful geocoded coordinates back to df_crimex

# Keep successful geocoding only
geo_success = geo_results[
    geo_results[["lat", "lon"]].notna().all(axis=1)
].copy()

print("Successful recovered coordinates:", len(geo_success))

# Attach geo_query to df_crimex missing rows
df_crimex["geo_query"] = None

df_crimex.loc[
    missing_geo.index,
    "geo_query"
] = missing_geo["geo_query"]

# Merge successful geocoding
df_crimex = df_crimex.merge(
    geo_success,
    on="geo_query",
    how="left"
)

# Fill only missing/invalid coordinates
fill_mask = (
    (df_crimex["is_valid_coordinate"] == 0)
    & df_crimex["lat"].notna()
    & df_crimex["lon"].notna()
)

df_crimex.loc[fill_mask, "latitude_final"] = df_crimex.loc[fill_mask, "lat"]
df_crimex.loc[fill_mask, "longitude_final"] = df_crimex.loc[fill_mask, "lon"]
df_crimex.loc[fill_mask, "is_valid_coordinate"] = 1
df_crimex.loc[fill_mask, "geo_precision_desc"] = "address_geocoded"

# For original valid coordinates
df_crimex["geo_precision_desc"] = df_crimex["geo_precision_desc"].fillna(
    "exact_coordinate"
)

# Drop temporary columns
df_crimex = df_crimex.drop(
    columns=["lat", "lon"],
    errors="ignore"
)

print("Rows filled:", fill_mask.sum())
print("Remaining invalid coordinates:", (df_crimex["is_valid_coordinate"] == 0).sum())
print("Shape:", df_crimex.shape)

Successful recovered coordinates: 452
Rows filled: 930
Remaining invalid coordinates: 1788
Shape: (1005372, 158)


In [43]:
# [6.7] Fix duplicated rows caused by duplicate geo_query matches

print("Current rows:", len(df_crimex))

# Rebuild successful geocoding as one row per geo_query
geo_success_unique = (
    geo_results[
        geo_results[["lat", "lon"]].notna().all(axis=1)
    ]
    .drop_duplicates(
        subset=["geo_query"],
        keep="first"
    )
    .copy()
)

print("Unique successful geo queries:", len(geo_success_unique))

# Restore from pre-merge version if available
# If you still have original df before merge as df_crimex_backup, use it.
# Otherwise remove duplicated case_id rows safely.

df_crimex = (
    df_crimex
    .drop_duplicates(
        subset=["case_id"],
        keep="first"
    )
    .copy()
)

print("Rows after fixing duplicates:", len(df_crimex))
print("Duplicate case_id:", len(df_crimex) - df_crimex["case_id"].nunique())

Current rows: 1005372
Unique successful geo queries: 359
Rows after fixing duplicates: 1004894
Duplicate case_id: 0


In [45]:
# [6.8] Safe coordinate fill using case_id, not dataframe index

# Recreate missing_geo from current df_crimex
missing_geo_current = df_crimex[
    df_crimex["is_valid_coordinate"] == 0
].copy()

# Rebuild geo_query safely
def build_address(row):
    parts = []

    if pd.notna(row.get("location")):
        parts.append(str(row["location"]))

    if pd.notna(row.get("cross_street")):
        parts.append(str(row["cross_street"]))

    if pd.notna(row.get("area_name")):
        parts.append(str(row["area_name"]))

    parts.append("Los Angeles USA")

    return ", ".join(parts)


missing_geo_current["geo_query"] = missing_geo_current.apply(
    build_address,
    axis=1
)

# Create lookup from successful geocoding
geo_success_unique = (
    geo_results[
        geo_results[["lat", "lon"]].notna().all(axis=1)
    ]
    .drop_duplicates(subset=["geo_query"], keep="first")
    .copy()
)

geo_lookup = geo_success_unique.set_index("geo_query")[["lat", "lon"]].to_dict("index")

# Add geo_query back using case_id mapping
geo_query_map = missing_geo_current.set_index("case_id")["geo_query"]

df_crimex["geo_query"] = df_crimex["case_id"].map(geo_query_map)

# Map coordinates
df_crimex["lat_geocoded"] = df_crimex["geo_query"].map(
    lambda x: geo_lookup.get(x, {}).get("lat")
)

df_crimex["lon_geocoded"] = df_crimex["geo_query"].map(
    lambda x: geo_lookup.get(x, {}).get("lon")
)

# Fill only invalid coordinates
fill_mask = (
    (df_crimex["is_valid_coordinate"] == 0)
    & df_crimex["lat_geocoded"].notna()
    & df_crimex["lon_geocoded"].notna()
)

df_crimex.loc[fill_mask, "latitude_final"] = df_crimex.loc[fill_mask, "lat_geocoded"]
df_crimex.loc[fill_mask, "longitude_final"] = df_crimex.loc[fill_mask, "lon_geocoded"]
df_crimex.loc[fill_mask, "is_valid_coordinate"] = 1

if "geo_precision_desc" not in df_crimex.columns:
    df_crimex["geo_precision_desc"] = "exact_coordinate"

df_crimex.loc[fill_mask, "geo_precision_desc"] = "address_geocoded"

# Cleanup temp columns
df_crimex = df_crimex.drop(
    columns=["lat_geocoded", "lon_geocoded"],
    errors="ignore"
)

print("Rows:", len(df_crimex))
print("Filled coordinates:", fill_mask.sum())
print("Remaining invalid:", (df_crimex["is_valid_coordinate"] == 0).sum())
print("Duplicate case_id:", len(df_crimex) - df_crimex["case_id"].nunique())

C:\Users\ayman\AppData\Local\Temp\ipykernel_109904\3779597269.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_crimex.loc[fill_mask, "latitude_final"] = df_crimex.loc[fill_mask, "lat_geocoded"]
C:\Users\ayman\AppData\Local\Temp\ipykernel_109904\3779597269.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_crimex.loc[fill_mask, "longitude_final"] = df_crimex.loc[fill_mask, "lon_geocoded"]


Rows: 1004894
Filled coordinates: 0
Remaining invalid: 1788
Duplicate case_id: 0


In [51]:
# [6.1] Final clean + save CRIMEX v1

from pathlib import Path
import json

FINAL_DIR = Path("data/final")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

DATASET_VERSION = "crimex_v1_la_incident_behavioral_intelligence"

parquet_path = FINAL_DIR / f"{DATASET_VERSION}.parquet"
csv_sample_path = FINAL_DIR / f"{DATASET_VERSION}_sample_50000.csv"
schema_path = FINAL_DIR / f"{DATASET_VERSION}_schema.json"

# -----------------------------------
# Final cleanup before saving
# -----------------------------------

df_final = df_crimex.copy()

# Remove temporary columns
df_final = df_final.drop(
    columns=[
        "geo_query"
    ],
    errors="ignore"
)

# Ensure geo precision is consistent
df_final["geo_precision_desc"] = df_final["geo_precision_desc"].fillna(
    "exact_coordinate"
)

# -----------------------------------
# Save full dataset
# -----------------------------------

df_final.to_parquet(
    parquet_path,
    index=False
)

# Save 50K sample
df_final.sample(
    n=50000,
    random_state=42
).to_csv(
    csv_sample_path,
    index=False
)

# Save schema
schema = {
    "dataset_version": DATASET_VERSION,
    "rows": int(df_final.shape[0]),
    "columns": int(df_final.shape[1]),
    "column_names": df_final.columns.tolist()
}

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(
        schema,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved full parquet:", parquet_path)
print("Saved sample:", csv_sample_path)
print("Saved schema:", schema_path)
print("Final shape:", df_final.shape)

Saved full parquet: data\final\crimex_v1_la_incident_behavioral_intelligence.parquet
Saved sample: data\final\crimex_v1_la_incident_behavioral_intelligence_sample_50000.csv
Saved schema: data\final\crimex_v1_la_incident_behavioral_intelligence_schema.json
Final shape: (1004894, 157)
